# 04 数据清洗：缺失、重复、类型、脏值、异常

清洗是数据分析里最费时间、也最能体现功力的一步。脏数据通常就这五类：

| 问题 | 典型表现 | 本章工具 |
| --- | --- | --- |
| 缺失 | 空单元格、`N/A`、`N/A`、`-` | `isna` / `dropna` / `fillna` |
| 重复 | 同一订单出现两次 | `duplicated` / `drop_duplicates` |
| 类型错 | 金额是 `"1,299.00"`、日期是文本 | `to_numeric` / `to_datetime` / `astype` |
| 脏文本 | 前后空格、`Beijing`/`北京`、全角字符 | `str.strip` / `map` / `normalize` |
| 异常值 | 数量为 -1、金额为 0、日期在 2050 年 | IQR / 业务规则 / `clip` |

本节用 `sales_orders_messy.csv`（故意做脏的 12 行订单）走一遍完整清洗。

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.unicode.east_asian_width", True)
pd.set_option("display.max_columns", 20)

DATA = Path("data")
if not DATA.exists():
    DATA = Path("python/pandas/data")

raw = pd.read_csv(DATA / "sales_orders_messy.csv")
print("洗之前：", raw.shape)
raw

洗之前： (12, 8)


,order_id,order_date,city,category,unit_price,qty,amount,sales_rep
0,S001,2024-01-05,北京,数码,"1,299.00",2.0,2598.0,张伟
1,S002,2024/01/07,上海,家居,199.5,1.0,199.5,李娜
2,S003,20240109,广州,服饰,¥ 89.9,3.0,269.7,王强
3,S004,2024-02-30,深圳,食品,12.5,NaN,12.5,赵敏
4,S005,2024-03-15 10:20,杭州,数码,899,2.0,1798.0,陈晨
5,S006,2024-03-16,北京,digital,1299,1.0,1299.0,张伟
6,S007,2024-03-18,成都,食品,9.9,-1.0,-9.9,刘洋
7,S007,2024-03-18,成都,食品,9.9,-1.0,-9.9,刘洋
8,NaN,2024-04-01,NaN,家居,299,1.0,299.0,李娜
9,S009,2024-04-05,上海,服饰,89.9,0.0,0.0,王强


## 1. 缺失值：先认清 pandas 里的几种"空"

| 写法 | dtype 里表现 | 说明 |
| --- | --- | --- |
| `np.nan` | float 列的 NaN | 数值缺失的事实标准 |
| `None` | 进 float 列变 NaN，进对象列变 None | Python 原生空值 |
| `pd.NaT` | 日期列的缺失 | Not a Time |
| `pd.NA` | `Int64` / `string` 等可空类型的缺失 | 显式的"空" |

`pd.isna()` 对这四种都返回 `True`，**这是判断缺失的统一入口**。

In [2]:
demo = pd.DataFrame({
    "float列": [1.0, np.nan],
    "文本列": ["a", None],
    "日期列": pd.to_datetime(["2024-01-01", None]),
    "可空整数": pd.array([1, pd.NA], dtype="Int64"),
    "空字符串": ["", "x"],
})
print(demo.dtypes.to_dict())
demo.isna()

{'float列': dtype('float64'), '文本列': <StringDtype(na_value=nan)>, '日期列': dtype('<M8[us]'), '可空整数': Int64Dtype(), '空字符串': <StringDtype(na_value=nan)>}


,float列,文本列,日期列,可空整数,空字符串
0,False,False,False,False,False
1,True,True,True,True,False


In [3]:
raw.isna().sum().to_frame("缺失数").assign(缺失比例=lambda d: (d["缺失数"] / len(raw)).round(2))

,缺失数,缺失比例
order_id,1,0.08
order_date,1,0.08
city,1,0.08
category,0,0.00
unit_price,1,0.08
qty,1,0.08
amount,1,0.08
sales_rep,0,0.00


### 删除缺失：`dropna`

| 参数 | 作用 |
| --- | --- |
| `how="any"`（默认） | 只要有一个空就删 |
| `how="all"` | 整行全空才删 |
| `subset=[...]` | 只看这几列 |
| `thresh=n` | 至少有 n 个非空才保留 |

In [4]:
print("默认 any        :", raw.dropna().shape)
print("只看 order_id   :", raw.dropna(subset=["order_id"]).shape)
print("至少 6 个非空   :", raw.dropna(thresh=6).shape)
print("全空才删        :", raw.dropna(how="all").shape)

默认 any        : (9, 8)
只看 order_id   : (11, 8)
至少 6 个非空   : (11, 8)
全空才删        : (12, 8)


### 填充缺失：`fillna` / `ffill` / `interpolate`

填充是**业务判断**，不是技术操作。金额缺失填 0 会拉低均值，填均值会抹平方差——
通常更好的是：能算出来就算（单价 × 数量），算不出来就留空，并在分析时显式处理。

In [5]:
f = pd.DataFrame({
    "city": ["北京", "北京", None, "上海", None],
    "amount": [100.0, np.nan, 300.0, np.nan, 500.0],
})
print("固定值填充:\n", f.fillna({"city": "未知", "amount": 0}))

print("\n按列分别填:")
print(f.assign(amount=f["amount"].fillna(f["amount"].mean())).round(1))

print("\n前向填充 ffill（用上一行的值补）:")
print(f["city"].ffill())

print("\n线性插值 interpolate:")
print(f["amount"].interpolate())

固定值填充:
    city  amount
0  北京   100.0
1  北京     0.0
2  未知   300.0
3  上海     0.0
4  未知   500.0

按列分别填:
   city  amount
0  北京   100.0
1  北京   300.0
2   NaN   300.0
3  上海   300.0
4   NaN   500.0

前向填充 ffill（用上一行的值补）:
0    北京
1    北京
2    北京
3    上海
4    上海
Name: city, dtype: str

线性插值 interpolate:
0    100.0
1    200.0
2    300.0
3    400.0
4    500.0
Name: amount, dtype: float64


## 2. 空字符串不是缺失

从 CSV / Excel 过来的数据，缺失经常以 `""`、`" "`、`"N/A"`、`"-"` 的形式出现。
`read_csv` 的 `na_values` 只能处理完全匹配的值，前后带空格的还得自己 `strip`。

In [6]:
text_demo = pd.DataFrame({"备注": ["", "  ", "N/A", "正常", None]})
print("原始缺失数:", text_demo["备注"].isna().sum())

# 先 strip，再把空串和占位符换成 pd.NA
cleaned = (
    text_demo["备注"]
    .astype("string")
    .str.strip()
    .replace({"": pd.NA, "N/A": pd.NA, "-": pd.NA})
)
print("处理后缺失数:", cleaned.isna().sum())
cleaned

原始缺失数: 1
处理后缺失数: 4


0    <NA>
1    <NA>
2    <NA>
3    正常
4    <NA>
Name: 备注, dtype: string

## 3. 重复：先定义"什么算同一行"

- **完全重复**：所有列都一样，直接 `drop_duplicates()`；
- **业务重复**：主键（如 `order_id`）相同但其他列有差异，这是数据冲突，要人工看，不能无脑删。

In [7]:
print("完全重复行数:", raw.duplicated().sum())
raw[raw.duplicated(keep=False)]

完全重复行数: 1


,order_id,order_date,city,category,unit_price,qty,amount,sales_rep
6,S007,2024-03-18,成都,食品,9.9,-1.0,-9.9,刘洋
7,S007,2024-03-18,成都,食品,9.9,-1.0,-9.9,刘洋


In [8]:
print("按 order_id 判断重复:", raw.duplicated(subset=["order_id"]).sum())
print("保留第一次出现:", len(raw.drop_duplicates(subset=["order_id"], keep="first")))
print("保留最后一次  :", len(raw.drop_duplicates(subset=["order_id"], keep="last")))
print("全标记出来    :", len(raw.drop_duplicates(subset=["order_id"], keep=False)))

按 order_id 判断重复: 1
保留第一次出现: 11
保留最后一次  : 11
全标记出来    : 10


## 4. 类型转换：把"看起来像数字"变成真数字

In [9]:
# 金额：¥ 符号 + 千分位 + 首尾空格，三种脏法叠加
price_text = raw["unit_price"].astype("string")
step1 = price_text.str.replace("¥", "", regex=False)
step2 = step1.str.replace(",", "", regex=False)
step3 = step2.str.strip()

converted = pd.to_numeric(step3, errors="coerce")   # 转不了的变 NaN，而不是报错
print(step3.tolist())
print(converted.tolist())

['1299.00', '199.5', '89.9', '12.5', '899', '1299', '9.9', '9.9', '299', '89.9', <NA>, '1299.00']
[1299.0, 199.5, 89.9, 12.5, 899.0, 1299.0, 9.9, 9.9, 299.0, 89.9, <NA>, 1299.0]


In [10]:
# 日期：格式混杂，用 format="mixed" 逐行猜；非法日期（2024-02-30）变 NaT
dates = pd.to_datetime(raw["order_date"], format="mixed", errors="coerce")
print(dates.tolist())
print("非法/缺失的日期数:", dates.isna().sum())

[Timestamp('2024-01-05 00:00:00'), Timestamp('2024-01-07 00:00:00'), Timestamp('2024-01-09 00:00:00'), NaT, Timestamp('2024-03-15 10:20:00'), Timestamp('2024-03-16 00:00:00'), Timestamp('2024-03-18 00:00:00'), Timestamp('2024-03-18 00:00:00'), Timestamp('2024-04-01 00:00:00'), Timestamp('2024-04-05 00:00:00'), NaT, Timestamp('2024-05-20 00:00:00')]
非法/缺失的日期数: 2


In [11]:
# 可空整数：qty 里有缺失，普通 int64 装不下，用 Int64
qty = pd.to_numeric(raw["qty"], errors="coerce")
print("float 版:", qty.tolist())
print("Int64 版:", qty.astype("Int64").tolist())   # 缺失显示为 <NA>，不是 NaN

float 版: [2.0, 1.0, 3.0, nan, 2.0, 1.0, -1.0, -1.0, 1.0, 0.0, 2.0, 1.0]
Int64 版: [2, 1, 3, <NA>, 2, 1, -1, -1, 1, 0, 2, 1]


In [12]:
# category：取值有限的列，转成 category 省内存、还能规定排序
city_cat = raw["city"].astype("string").str.strip().astype("category")
print("类别:", city_cat.cat.categories.tolist())
print("编码:", city_cat.cat.codes.tolist())

类别: ['上海', '北京', '广州', '成都', '杭州', '深圳']
编码: [1, 0, 2, 5, 4, 1, 3, 3, -1, 0, 2, 1]


## 5. 脏文本：空格、大小写、同义词、全角字符

顺序很重要：**先归一化（去空格 / 全角转半角 / 大小写），再映射同义词**，否则
`"Beijing"` 和 `"beijing "` 会被当成两个值。

In [13]:
# 全角 → 半角：str.normalize("NFKC") 是处理中文全角最省事的办法
fullwidth = pd.Series(["ＡＢＣ１２３", "（测试）"])
print(fullwidth.str.normalize("NFKC").tolist())

['ABC123', '(测试)']


In [14]:
# 城市同义词统一
city_map = {"Beijing": "北京", "beijing": "北京", "SH": "上海", "上海 ": "上海"}
city_clean = (
    raw["city"].astype("string").str.strip().str.normalize("NFKC")
    .replace(city_map)
)
city_clean.value_counts(dropna=False)

city
北京    3
上海    2
广州    2
成都    2
深圳    1
杭州    1
<NA>    1
Name: count, dtype: int64[pyarrow]

In [15]:
# 品类同义词统一；大小写不一致先用 str.lower 再映射
cat_clean = (
    raw["category"].astype("string").str.strip()
    .replace({"digital": "数码", "Digital": "数码", "DIGITAL": "数码"})
)
cat_clean.value_counts(dropna=False)

category
数码    4
食品    4
家居    2
服饰    2
Name: count, dtype: int64[pyarrow]

## 6. 异常值：统计规则 + 业务规则

两条路一起用：

- **统计规则**：IQR（四分位距）法，超出 `[Q1 - 1.5*IQR, Q3 + 1.5*IQR]` 的极端值；
- **业务规则**：数量必须 ≥ 1、金额必须 > 0、日期不能在未来。

注意：统计规则只负责"标记"，删不删要结合业务。金额本来就是长尾分布时，删掉大额订单是灾难。

In [16]:
# 先用一批"看起来正常 + 一个明显离群"的小数据，看清 IQR 规则怎么算
demo_amount = pd.Series([10, 12, 11, 13, 12, 500])
q1, q3 = demo_amount.quantile([0.25, 0.75])
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(f"Q1={q1} Q3={q3} IQR={iqr} 正常区间=[{lower:.1f}, {upper:.1f}]")
print("被判为异常的值:", demo_amount[(demo_amount < lower) | (demo_amount > upper)].tolist())

Q1=11.25 Q3=12.75 IQR=1.5 正常区间=[9.0, 15.0]
被判为异常的值: [500]


In [17]:
# 再把规则用到真实订单上
amount_num = pd.to_numeric(raw["amount"], errors="coerce")
q1, q3 = amount_num.quantile([0.25, 0.75])
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(f"真实订单：Q1={q1:.1f} Q3={q3:.1f} IQR={iqr:.1f} 正常区间=[{lower:.1f}, {upper:.1f}]")

outliers = amount_num[(amount_num < lower) | (amount_num > upper)]
print("统计意义上的异常值:", outliers.tolist())
print("→ 这批数据没有极端值。IQR 只是'标记'工具，删不删要结合业务判断。")

真实订单：Q1=6.2 Q3=1299.0 IQR=1292.8 正常区间=[-1932.9, 3238.1]
统计意义上的异常值: []
→ 这批数据没有极端值。IQR 只是'标记'工具，删不删要结合业务判断。


In [18]:
# 业务规则：金额和数量必须为正
qty_num = pd.to_numeric(raw["qty"], errors="coerce")
bad_rows = raw[(amount_num <= 0) | (qty_num <= 0)]
bad_rows[["order_id", "qty", "amount"]]

,order_id,qty,amount
6,S007,-1.0,-9.9
7,S007,-1.0,-9.9
9,S009,0.0,0.0


In [19]:
# clip：把超出范围的值"压"到边界，而不是删除
print(amount_num.clip(lower=0, upper=3000).tolist())

[2598.0, 199.5, 269.7, 12.5, 1798.0, 1299.0, 0.0, 0.0, 299.0, 0.0, nan, 1299.0]


## 7. 组装一条清洗管线

把上面的零散操作固定成函数：**每一步只做一件事，最后统计每步删/改了多少行**。
这是工程上最实用的形态——可复用、可解释、可回滚。

In [20]:
def clean_orders(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
    """把脏订单表清洗成可分析的表，同时返回一份清洗报告。"""
    report = {"原始行数": len(df)}
    df = df.copy()

    # 1) 主键：去空格、空串置空
    df["order_id"] = df["order_id"].astype("string").str.strip()
    df["order_id"] = df["order_id"].replace({"": pd.NA})

    # 2) 文本列去空格 + 同义词归一
    df["city"] = df["city"].astype("string").str.strip().replace({"Beijing": "北京", "beijing": "北京"})
    df["city"] = df["city"].replace({"": pd.NA})
    df["category"] = df["category"].astype("string").str.strip().replace({"digital": "数码", "Digital": "数码"})
    df["sales_rep"] = df["sales_rep"].astype("string").str.strip().replace({"": pd.NA})

    # 3) 金额：去符号/千分位 → 数字
    price = (
        df["unit_price"].astype("string")
        .str.replace("¥", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.strip()
    )
    df["unit_price"] = pd.to_numeric(price, errors="coerce")

    # 4) 数量、日期
    df["qty"] = pd.to_numeric(df["qty"], errors="coerce").astype("Int64")
    df["order_date"] = pd.to_datetime(df["order_date"], format="mixed", errors="coerce")

    # 5) 金额缺失时用 单价 × 数量 补算
    df["amount"] = pd.to_numeric(df["amount"], errors="coerce")
    computed = (df["unit_price"] * df["qty"]).round(2)
    df["amount"] = df["amount"].fillna(computed)

    # 6) 完全重复行去重（要在类型归一之后做，否则"看着一样"的两行可能识别不出来）
    before = len(df)
    df = df.drop_duplicates(keep="first")
    report["删除重复行"] = before - len(df)

    # 7) 业务规则：主键为空、日期非法、数量/金额非正 → 丢弃
    before = len(df)
    df = df.dropna(subset=["order_id", "order_date"])
    valid = (df["qty"].fillna(0) > 0) & (df["amount"].fillna(0) > 0)   # 可空布尔要先补 False
    df = df[valid]
    report["按业务规则丢弃"] = before - len(df)

    # 8) 校验：金额是否等于 单价 × 数量（允许 0.01 的浮点误差）
    mismatch = ~np.isclose(df["amount"], df["unit_price"] * df["qty"], atol=0.01)
    report["金额与单价×数量不符"] = int(mismatch.sum())

    report["最终行数"] = len(df)
    df = df.reset_index(drop=True)
    return df, pd.Series(report)


clean, report = clean_orders(raw)
print(report.to_string())
clean

原始行数               12
删除重复行              1
按业务规则丢弃          5
金额与单价×数量不符     0
最终行数                6


,order_id,order_date,city,category,unit_price,qty,amount,sales_rep
0,S001,2024-01-05 00:00:00,北京,数码,1299.0,2,2598.0,张伟
1,S002,2024-01-07 00:00:00,上海,家居,199.5,1,199.5,李娜
2,S003,2024-01-09 00:00:00,广州,服饰,89.9,3,269.7,王强
3,S005,2024-03-15 10:20:00,杭州,数码,899.0,2,1798.0,陈晨
4,S006,2024-03-16 00:00:00,北京,数码,1299.0,1,1299.0,张伟
5,S011,2024-05-20 00:00:00,北京,数码,1299.0,1,1299.0,张伟


In [21]:
clean.dtypes

order_id              string
order_date    datetime64[us]
city                  string
category              string
unit_price           Float64
qty                    Int64
amount               Float64
sales_rep             string
dtype: object

## 8. 小练习

用清洗后的表回答：

1. 每个城市的订单数和总金额（按总金额降序）；
2. 找出金额最高的那一单，显示它的 `order_id`、`city`、`amount`；
3. 在上面的城市汇总里加一列 `客单价 = 总金额 / 订单数`，保留 2 位小数。

In [22]:
city_summary = clean.groupby("city", as_index=False).agg(
    订单数=("order_id", "count"),
    总金额=("amount", "sum"),
).sort_values("总金额", ascending=False).reset_index(drop=True)
city_summary

,city,订单数,总金额
0,北京,3,5196.0
1,杭州,1,1798.0
2,广州,1,269.7
3,上海,1,199.5


In [23]:
top = clean.nlargest(1, "amount")[["order_id", "city", "amount"]]
top

,order_id,city,amount
0,S001,北京,2598.0


In [24]:
city_summary.assign(客单价=(city_summary["总金额"] / city_summary["订单数"]).round(2))

,city,订单数,总金额,客单价
0,北京,3,5196.0,1732.0
1,杭州,1,1798.0,1798.0
2,广州,1,269.7,269.7
3,上海,1,199.5,199.5


## 9. 小结与检查清单

每次清洗完，用这张清单自检：

- [ ] `df.info()` 里每列 dtype 对不对（日期是 `datetime64`、金额是 `float`、ID 是字符串）
- [ ] 缺失值数量是否符合预期，缺失是否被误填成 0 或 `"nan"`
- [ ] 主键是否唯一：`df["id"].is_unique`
- [ ] 分类列的取值是否只有预期的几类：`value_counts()`
- [ ] 数值列的范围是否合理：`describe()`
- [ ] 行数变化能解释：删了多少、为什么删

记住三条原则：

1. **先看一眼再动手**：`value_counts()` 比任何猜测都可靠。
2. **能推导就推导，不要凭空填**：金额 = 单价 × 数量，比填均值靠谱。
3. **清洗要留痕**：每一步改了多少行，写进报告里，否则出问题无法回溯。